# Feature Engineering & Data Quality
### Titanic: Machine Learning from Disaster

**Objective:** Improve the predictive signal for passenger survival through hypothesis-driven feature engineering, and audit the dataset for quality issues that could make a model unreliable or biased.

**Target:** `Survived` (1 = survived, 0 = did not survive), a binary classification task.

**Dataset:** Kaggle `train.csv`, 891 passengers. Kaggle's `test.csv` is not used because it has no labels; instead, a held-out test set is created from `train.csv`.

**Notebook structure**
1. Setup & data loading
2. Data-quality audit
3. Leakage & class imbalance checks
4. Train/test split & baseline pipeline
5. Feature engineering (with hypotheses)
6. Engineered model
7. Before/after comparison & hypothesis evaluation
8. Conclusion

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, f1_score, classification_report

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv("data/train.csv")
print("Shape:", df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


### Column dictionary

| Column | Meaning |
|---|---|
| PassengerId | Row identifier (no predictive meaning) |
| Survived | Target: 1 = survived, 0 = died |
| Pclass | Ticket class (1 = upper, 2 = middle, 3 = lower) |
| Name | Full name, including title (Mr, Mrs, Miss, Master…) |
| Sex | male / female |
| Age | Age in years |
| SibSp | Number of siblings/spouses aboard |
| Parch | Number of parents/children aboard |
| Ticket | Ticket number |
| Fare | Ticket price |
| Cabin | Cabin number |
| Embarked | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

## 2. Data-Quality Audit

Before modelling, we check whether the data can be trusted. Each check below is followed by what we actually found in *this* dataset.

### 2.1 Missing values

In [3]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
})
missing[missing.missing_count > 0]

,missing_count,missing_pct
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


#### 2.1a Is `Age` missing at random? Compare passengers with and without an age.

In [4]:
df["Age_missing"] = df["Age"].isna()

print("Share of each ticket class, by whether Age is missing:")
display(df.groupby("Age_missing")["Pclass"].value_counts(normalize=True).unstack().round(3))

print("Share of Age missing inside each class:")
display(df.groupby("Pclass")["Age_missing"].mean().round(3))

print("Survival rate by whether Age is missing:")
display(df.groupby("Age_missing")["Survived"].mean().round(3))

Share of each ticket class, by whether Age is missing:


Pclass,1,2,3
Age_missing,,,
False,0.261,0.242,0.497
True,0.169,0.062,0.768


Share of Age missing inside each class:


Pclass
1    0.139
2    0.060
3    0.277
Name: Age_missing, dtype: float64

Survival rate by whether Age is missing:


Age_missing
False    0.406
True     0.294
Name: Survived, dtype: float64

**Finding 1 — `Age` is missing for 177 passengers (19.9%), and the missingness is *not random*.**
- 76.8% of passengers with a missing age travelled in 3rd class, compared with 49.7% of passengers whose age is known. Age is missing for 27.7% of 3rd-class passengers but only 13.9% of 1st class and 6.0% of 2nd class.
- Survival also differs: 29.4% for passengers with a missing age vs 40.6% for those with a known age.

**Why it matters:** the records with missing ages come disproportionately from the group with the lowest survival. Filling them all with one overall median treats them as "typical" passengers and can hide this pattern. Later, the imputer will be fitted inside the Pipeline (training data only), and we will test an `AgeMissing` flag to see whether the missingness itself carries signal.

#### 2.1b Is `Cabin` missing at random?

In [6]:
df["Cabin_missing"] = df["Cabin"].isna()

print("Share of each class, by whether Cabin is missing:")
display(df.groupby("Cabin_missing")["Pclass"].value_counts(normalize=True).unstack().round(3))

print("Survival rate by whether Cabin is missing:")
display(df.groupby("Cabin_missing")["Survived"].mean().round(3))

print("Survival rate by class AND whether Cabin is missing:")
display(df.groupby(["Pclass", "Cabin_missing"])["Survived"].agg(["mean", "size"]).round(3))

Share of each class, by whether Cabin is missing:


Pclass,1,2,3
Cabin_missing,,,
False,0.863,0.078,0.059
True,0.058,0.245,0.697


Survival rate by whether Cabin is missing:


Cabin_missing
False    0.667
True     0.300
Name: Survived, dtype: float64

Survival rate by class AND whether Cabin is missing:


mean  size
Pclass Cabin_missing             
1      False          0.665   176
       True           0.475    40
2      False          0.812    16
       True           0.440   168
3      False          0.500    12
       True           0.236   479

**Finding 2 — `Cabin` is missing for 687 passengers (77.1%), and *having* a cabin value is tied to class and survival.**
- 86.3% of passengers with a recorded cabin were 1st class; 69.7% of passengers with no cabin were 3rd class.
- Survival is 66.7% when a cabin is recorded vs 30.0% when it is not.
- Part of this is just class (upper-class passengers had recorded cabins and survived more). But the gap persists *inside* each class: in 2nd class, 81.3% survived with a cabin vs 44.0% without; in 1st class, 66.5% vs 47.5%. (Some cells are small, such as only 16 second-class passengers with a cabin.)

**Why it matters:** with 77% missing, `Cabin` cannot be used as a normal column. Its only usable signal is "was a cabin recorded at all?". That may reflect how the records were *collected* (cabin details were more likely to be known for survivors and upper-class passengers) rather than a genuine cause of survival. That is a possible leakage-style risk, which we test carefully in section 3.

#### 2.1c The 2 missing `Embarked` values

In [8]:
df[df["Embarked"].isna()][["PassengerId", "Name", "Pclass", "Ticket", "Fare", "Cabin", "Embarked", "Survived"]]

,PassengerId,Name,Pclass,Ticket,Fare,Cabin,Embarked,Survived
61,62,"Icard, Miss. Amelie",1,113572,80.0,B28,NaN,1
829,830,"Stone, Mrs. George Nelson (Martha Evelyn)",1,113572,80.0,B28,NaN,1


Both passengers share the same ticket (113572), fare (80.0) and cabin (B28), so they travelled together and boarded at the same port. With only 2 of 891 rows affected, filling them with the most frequent port inside the Pipeline is a reasonable, low-risk fix.891 rows affected, filling them with the most frequent port inside the Pipeline is a reasonable, low-risk fix.